# dataclasses-replace-args — faded example 2: Merge defaults under per-run overrides before replace

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataclasses-replace-args`. The last cell reports your progress on the `Config: dataclasses.replace args` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: dataclasses.replace args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataclasses-replace-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataclasses-replace-args"
DD_SUBTOPIC = "Config: dataclasses.replace args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Often a sweep has shared defaults plus per-run overrides where the run wins on conflict. You merge the two dicts (run override last so it takes precedence) and splat the merged dict into `dataclasses.replace(base, **merged)`. Base stays untouched and the merged dict drives a single clone.

## Faded exercise 2

### Faded — merge shared defaults under a run override, then replace

Implement `apply_with_defaults(base, defaults, override)`. Both `defaults` and `override` are dicts of field overrides; on a key collision the `override` value must win. Merge them (override last), then return ONE new `TrainingArgs` from base with the merged overrides applied. Complete the blanked line that produces the merged override dict.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
from dataclasses import dataclass, replace

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')

def apply_with_defaults(base, defaults, override):
    merged = {**defaults, **override}
    return replace(base, **merged)


def _test():
    base = TrainingArgs()
    defaults = {'lr': 1e-4, 'epochs': 5}
    override = {'lr': 9e-4, 'batch_size': 64}
    v = apply_with_defaults(base, defaults, override)
    assert isinstance(v, TrainingArgs)
    # override wins on lr
    assert v.lr == 9e-4
    # default applied where no collision
    assert v.epochs == 5
    # override-only field applied
    assert v.batch_size == 64
    # untouched field from base
    assert v.optimizer_name == base.optimizer_name
    assert v is not base
    # base not mutated
    assert base.lr == 1e-3 and base.epochs == 10 and base.batch_size == 32
    # empty dicts -> faithful clone
    clone = apply_with_defaults(base, {}, {})
    assert clone is not base
    assert (clone.lr, clone.batch_size, clone.epochs) == (base.lr, base.batch_size, base.epochs)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from dataclasses import dataclass, replace

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')

def apply_with_defaults(base, defaults, override):
    merged = {**defaults, **override}
    return replace(base, **merged)
```
</details>